Ce notebook permet de créer le contexte nécessaire au LLM pour faire une analyse morphologique la plus précise et complète possible. Dans un premier temps, le contexte est composé du lemme, de sa catégorie, sa définition et son étymologie. Chaque information vient de Kaikki.

Dans un deuxième temps nous essaierons d'intégrer une "famille dérivationnelle" pour chaque entrée. Les informations seront extraites en partie de Kaikki, pour les entrées qui ont des rubriques intéressantes, et de Démonette. Toutes les entrées supplétives ont une famille dérivationnelle dans Démonette car elles ont été préalablement sélectionnées dans Démonette.

# Installation et configuration

In [ ]:
# Uniquement pour Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Uniquement pour Colab
%cd '/content/drive/MyDrive/FAC/Master/M2/Stage/Code/Dataset'
# path Maïwenn à changer

/content/drive/MyDrive/FAC/Master/M2/Stage/Code/Dataset


In [ ]:
import pandas as pd
import json
import ast

# Préparation du contexte avec Kaikki

In [ ]:
chunks = pd.read_json("./fr-extract.jsonl.gz", lines=True, chunksize=100)

entries = pd.read_csv("./selected_entries.csv", usecols=["lemma", "cat"])

lemma_list = entries["lemma"].tolist()
cat_list = entries["cat"].tolist()
entries_list = list(zip(lemma_list, cat_list))

full_df = pd.DataFrame()

for chunk in chunks :
  try :
    context = chunk[chunk["lang"] == "Français"][["word", "pos_title", "senses", "etymology_texts"]]
  except Exception as e :
    #print(e)
    #print(chunk)
    continue
  # problème des homographes :
  # on crée une liste de couples (Mot, Catégorie) à partir des mots que j'ai sélectionnés,
  # si j'ai sélectionné le verbe lire alors j'aurai le couple (lire, Verbe) dans ma liste
  # ainsi dans Kaikki je n'extrais que les informations du verbe lire et non pas du nom lire
  context["word_pos_title"] = list(zip(context["word"].tolist(), context["pos_title"].tolist()))
  context = context[context["word_pos_title"].isin(entries_list)]

  if len(context) == 0 :
    continue

  print("Found : ", "; ".join(context["word"]) )

  # on fait une liste de len(context) listes vides
  deriv_fam = [ [] for i in range(len(context)) ]

  # on récupère les mots qui font partie de la famille grâce à certaines rubriques Kaikki (derived, paronyms, synonyms, related...)
  for enum_idx, df_idx in enumerate(context.index) :
    for col in ["derived", "paronyms", "synonyms", "related", "antonyms", "holonyms"] :
      if col in chunk.columns :
        row = chunk.loc[df_idx, col]
        if isinstance(row, list) : # si c'est NaN alors ça ne sera pas une liste (dans les cas où la rubrique est vide)
          for dictionary in row :
            if len(dictionary["word"].split()) == 1 : # on ne prend que 1 mot, je ne veux pas "arme de siège" ou "canot du nord" dans ma famille
              deriv_fam[enum_idx].append( dictionary["word"] )

    # on enlève de la famille tous les mots qui ont un trait d'union car je ne veux pas "bras-le-corps" ou "mont-de-piété"
    i = 0

    while i < len(deriv_fam) :
      w = 0
      while w < len(deriv_fam[i]) :
        if len(deriv_fam[i][w].split("-")) >= 2 :
          deriv_fam[i].remove(deriv_fam[i][w])
        else :
          w += 1
      i += 1

  context["famille"] = deriv_fam

  context["senses"] = context["senses"].apply( lambda x: x[0]["glosses"] if isinstance(x, list) and len(x) > 0 and "glosses" in x[0] else None )
  context = context.rename(columns={"word": "mot", "pos_title": "catégorie", "senses": "définition", "etymology_texts": "étymologie"})
  context = context.drop(columns=["word_pos_title"])

  if len(full_df) == 0 :
    full_df = context
    full_df.to_csv("./context.csv", index=False)
  else :
    full_df = pd.concat( [full_df, context], axis=0 )
    full_df.to_csv("./context.csv", index=False)

  if len(full_df) >= len(entries_list) :
    break

In [ ]:
# pour vérifier si tous les mots que j'ai sélectionnés sont bien dans le csv final
lemma_list
mot_list = full_df["mot"].tolist()

for lemma in lemma_list :
  if lemma not in mot_list :
    print(lemma)

# Intégration famille Démonette

In [ ]:
# code très très long à tourner
families = pd.read_csv("./families.csv", sep="\t")
lexemes = pd.read_csv("./lexemes.csv", sep="\t")

def find_fid(lemma, category, lexemes) :
  for idx, row in lexemes.iterrows() :
    if lemma == row["graphie"] and category[0] == row["cat"][0] :
      return row["fid"]

def find_lids(fid, families) :
  for idx, row in families.iterrows() :
    if fid == row["fid"] :
      return row["lids"].split(";")

def find_lemma(lid, lexemes) :
  for idx, row in lexemes.iterrows() :
    if lid == row["lid"] :
      return row["graphie"]

def find_family(lemma, category, lexemes, families) :
  family = []
  fid = find_fid(lemma, category, lexemes)
  lids = find_lids(fid, families)
  for lid in lids :
    lemma = find_lemma(lid, lexemes)
    family.append(lemma)
  return family

for idx, row in full_df.iterrows() :
  full_df.loc[idx,"famille"] = row["famille"] + find_family(row["mot"], row["catégorie"], lexemes, families)
  full_df.to_csv("./context.csv", index=False)

/tmp/ipykernel_10999/1698942431.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  lexemes = pd.read_csv("./lexemes.csv", sep="\t")


In [13]:
# on enlève les doublons au sein de la famille (car des mots apparaissent à la fois dans Kaikki et dans Démonette)
full_df = pd.read_csv("./context.csv")
full_df["famille"] = full_df["famille"].apply(lambda x: list(dict.fromkeys(ast.literal_eval(x))) if isinstance(x, str) else list(dict.fromkeys(x)))
full_df.to_csv("./context.csv", index=False)